In [0]:
from pyspark.sql.functions import col, count, when, isnan, isnull, countDistinct, lit, trim, year, month
from pyspark.errors import AnalysisException
import re
import pandas as pd

In [0]:
df = spark.table("project.cambridgeshire_data_raw.combined_raw")

In [0]:
# Drop Context and Falls within columns - context is 100% null and Falls within is the equivalent of Reported by
df = df.drop("context", "falls_within")

print(f"Columns after dropping: {df.columns}")
print(f"Total columns: {len(df.columns)}")

In [0]:
df = df.dropna(subset=['lsoa_code', 'lsoa_name', 'longitude', 'latitude'])

print(f"{df.count():,} rows")

In [0]:
# Check if null values in crime_id and last_outcome_category correspond to anti-social behaviour

print("NULL VALUES AND CRIME TYPE")

# Check records with null crime_id
null_crime_id = df.filter(col("crime_id").isNull())
print(f"Total records with null crime_id: {null_crime_id.count():,}")

print("\nCrime types for null crime_id:")
null_crime_id.groupBy("crime_type").count().orderBy(col("count").desc()).show(truncate=False)

# Check records with null last_outcome_category
null_outcome = df.filter(col("last_outcome_category").isNull())
print(f"Total records with null last_outcome_category: {null_outcome.count():,}")

print("\nCrime types for null last_outcome_category:")
null_outcome.groupBy("crime_type").count().orderBy(col("count").desc()).show(truncate=False)

# Check records with BOTH nulls
both_null = df.filter(col("crime_id").isNull() & col("last_outcome_category").isNull())
print(f"Total records with both fields null: {both_null.count():,}")

print("\nCrime types for both fields null:")
both_null.groupBy("crime_type").count().orderBy(col("count").desc()).show(truncate=False)

# Check anti-social behaviour records
asb_records = df.filter(col("crime_type") == "Anti-social behaviour")
total_asb = asb_records.count()
print(f"Total Anti-social behaviour records: {total_asb:,}")

asb_with_crime_id = asb_records.filter(col("crime_id").isNotNull()).count()
asb_without_crime_id = asb_records.filter(col("crime_id").isNull()).count()

print(f"  With crime_id: {asb_with_crime_id:,} ({(asb_with_crime_id/total_asb)*100:.2f}%)")
print(f"  Without crime_id (null): {asb_without_crime_id:,} ({(asb_without_crime_id/total_asb)*100:.2f}%)")

In [0]:
# Replace null entries in crime_id and last_outcome_category

replacement_text = "Anti-social behaviour is not considered a crime"

#create null counts of before and after for comparison - check for difference in null counts
crime_id_nulls_before = df.filter(col("crime_id").isNull()).count()
outcome_nulls_before = df.filter(col("last_outcome_category").isNull()).count()
print(f"Null values in crime_id: {crime_id_nulls_before:,}")
print(f"Null values in last_outcome_category: {outcome_nulls_before:,}")

df = df.fillna({
    "crime_id": replacement_text,
    "last_outcome_category": replacement_text
})

print("\nCheck null values after replacement")
crime_id_nulls_after = df.filter(col("crime_id").isNull()).count()
outcome_nulls_after = df.filter(col("last_outcome_category").isNull()).count()
print(f"Null values in crime_id: {crime_id_nulls_after:,}")
print(f"Null values in last_outcome_category: {outcome_nulls_after:,}")

In [0]:
# Get distinct entries for month, crime_type, and last_outcome_category - check for outliers

# 1. Distinct Months
print("\nDISTINCT MONTHS")
months = df.select("month").distinct().orderBy("month").collect()
print(f"Total distinct months: {len(months)}")
print("\nMonths:")
for row in months:
    print(f"  {row['month']}")

# 2. Distinct Crime Types
print("\nDISTINCT CRIME TYPES")
crime_types = df.select("crime_type").distinct().orderBy("crime_type").collect()
print(f"Total distinct crime types: {len(crime_types)}")
print("\nCrime Types:")
for row in crime_types:
    print(f"  {row['crime_type']}")

# 3. Distinct Last Outcome Categories
print("\nDISTINCT LAST OUTCOME CATEGORIES")
outcomes = df.select("last_outcome_category").distinct().orderBy("last_outcome_category").collect()
print(f"Total distinct outcome categories: {len(outcomes)}")
print("\nOutcome Categories:")
for row in outcomes:
    print(f"  {row['last_outcome_category']}")

In [0]:
# Trim whitespace from crime_type and last_outcome_category columns

df = df.withColumn("crime_type", trim(col("crime_type"))) \
       .withColumn("last_outcome_category", trim(col("last_outcome_category")))


In [0]:
# Separate month column into year and month columns

df = df.withColumn("year", year(col("month"))) \
       .withColumn("month_num", month(col("month")))

# Drop the original month column
df = df.drop("month")

print("\nNew year and month_num columns (sample):")
df.select("year", "month_num").show(10)
print(f"\nUpdated columns: {df.columns}")

In [0]:
# Add month name column based on month_num

df = df.withColumn("month_name", 
    when(col("month_num") == 1, "January")
    .when(col("month_num") == 2, "February")
    .when(col("month_num") == 3, "March")
    .when(col("month_num") == 4, "April")
    .when(col("month_num") == 5, "May")
    .when(col("month_num") == 6, "June")
    .when(col("month_num") == 7, "July")
    .when(col("month_num") == 8, "August")
    .when(col("month_num") == 9, "September")
    .when(col("month_num") == 10, "October")
    .when(col("month_num") == 11, "November")
    .when(col("month_num") == 12, "December")
    .otherwise("Unknown")
)

print("\nSample of year, month_num, and month_name:")
df.select("year", "month_num", "month_name").show(15)
print(f"\nUpdated columns: {df.columns}")

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("project.cambridgeshire_data_clean.combined_clean")